# Semantic CV-to-Job Matcher — Exploration

A quick, hands-on look at the pipeline this project's `app.py` runs, using the
SAME `src/` modules the app uses (not a reimplementation):

1. Load the embedding model via `src/embeddings/service.py`
2. Embed a toy job description + a few CV snippets
3. Inspect pairwise cosine-similarity scores
4. Sanity-check `src/matching/ranker.py`'s ranking order

This notebook needs internet on its FIRST run only (to download
`paraphrase-MiniLM-L3-v2`, ~61MB) — every run after that is fully offline.
See `src/embeddings/service.py`'s module docstring for why.

In [ ]:
import sys
from pathlib import Path

# Make the project root importable (this notebook lives in notebooks/, one
# level below the project root where config.py and src/ live) so we can
# import the SAME src/ modules app.py uses, rather than duplicating logic.
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd

from src.embeddings.service import embed
from src.matching.ranker import cosine_similarity, rank_cvs

print("Imports OK — no model downloaded yet (embed() loads it lazily on first call).")

## 1. Toy Job Description + CV Snippets

One job description and three CV snippets, picked so the ranking result is
obvious to a human reader before we even run the model — a strong CV match,
a weak/unrelated match, and a partial match. That makes it easy to sanity-check
the model's output against intuition.

In [ ]:
job_description = (
    "We are hiring a Backend Engineer with strong Python experience, "
    "comfortable building REST APIs with FastAPI, working with PostgreSQL, "
    "and deploying services with Docker."
)

cv_snippets = {
    "Strong Match CV": (
        "Software engineer with 4 years building REST APIs in Python using "
        "FastAPI and Flask, PostgreSQL database design, and containerized "
        "deployments with Docker and Kubernetes."
    ),
    "Partial Match CV": (
        "Full-stack developer experienced with React and Node.js, some "
        "exposure to Python scripting and SQL databases for reporting."
    ),
    "Unrelated CV": (
        "Graphic designer specializing in brand identity, illustration, "
        "and print layout using Adobe Photoshop and Illustrator."
    ),
}

print(f"Job description:\n  {job_description}\n")
for name, text in cv_snippets.items():
    print(f"{name}:\n  {text}\n")

## 2. Embed Everything

`embed()` is the ONLY function in this project that calls
`SentenceTransformer.encode(...)` (see `src/embeddings/service.py`'s
HIGHLIGHTS). We call it once for the job description and once for all CV
snippets together — the first call in this notebook is what triggers the
one-time model download.

In [ ]:
job_vec = embed([job_description])[0]
cv_names = list(cv_snippets.keys())
cv_vecs = embed(list(cv_snippets.values()))

print(f"job_vec shape: {job_vec.shape}")
print(f"cv_vecs shape: {cv_vecs.shape}")

## 3. Pairwise Cosine Similarity

Using `src/matching/ranker.py`'s hand-rolled `cosine_similarity()` — the same
`(a . b) / (||a|| * ||b||)` math from the Week 2 NumPy dot-product lecture,
just applied to sentence vectors instead of raw numbers.

In [ ]:
scores = {name: cosine_similarity(job_vec, cv_vecs[i]) for i, name in enumerate(cv_names)}

scores_df = pd.DataFrame(
    {"CV": list(scores.keys()), "Cosine Similarity": list(scores.values())}
).sort_values("Cosine Similarity", ascending=False).reset_index(drop=True)

scores_df

**Sanity check:** the "Strong Match CV" (Python/FastAPI/PostgreSQL/Docker —
the same concepts as the job description, in different words) should score
clearly highest, "Unrelated CV" (graphic design) clearly lowest, and
"Partial Match CV" somewhere in between. This is exactly the "semantic, not
keyword" matching this project exists to demonstrate — none of the CVs reuse
the job description's exact phrasing.

## 4. Sanity-Check `rank_cvs()`

Now run the SAME `rank_cvs()` function `app.py` calls on the Match tab, and
confirm its output (sorted `MatchResult` list, 1-based `rank`) agrees with
the manual similarity table above.

In [ ]:
results = rank_cvs(job_vec, cv_vecs, cv_names)

for r in results:
    print(f"#{r.rank}  {r.name:<20s}  score={r.score:.4f}")

assert results[0].name == "Strong Match CV", "Expected the strong match to rank #1"
assert results[-1].name == "Unrelated CV", "Expected the unrelated CV to rank last"
print("\nRanking matches intuition \u2713")

## Takeaways

- `embed()` and `rank_cvs()` — the exact functions `app.py` uses — produce a
  ranking that matches human intuition on an easy toy example, without any
  of the CV snippets reusing the job description's exact wording.
- Cosine similarity stayed well-behaved (no NaNs, no magnitude bias) even
  though the snippets vary noticeably in length.
- This is the same pipeline `app.py`'s **Match** tab runs on real
  pasted/uploaded text — this notebook is just a small, fast, offline-after-
  first-run way to inspect it outside the Streamlit UI.